# 📈 Stock Price Movement Prediction (Kaggle Style)
Classification of next-day direction using historical features with LightGBM + SHAP + Gradio UI.

In [ ]:
!pip install yfinance lightgbm shap gradio --quiet

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import shap
import gradio as gr


In [ ]:
df = yf.download("AAPL", start="2022-01-01", end="2023-12-31")
df["return"] = df["Close"].pct_change()
df["volatility"] = df["return"].rolling(5).std()
df["momentum"] = df["Close"] - df["Close"].rolling(5).mean()
df["next_direction"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df.dropna(inplace=True)


In [ ]:
X = df[["Open", "High", "Low", "Close", "Volume", "volatility", "momentum"]]
y = df["next_direction"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
model = lgb.LGBMClassifier()
model.fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values[1], X_test)


In [ ]:
def predict_stock_movement(open_p, high, low, close, volume, volatility, momentum):
    row = pd.DataFrame([[open_p, high, low, close, volume, volatility, momentum]], columns=X.columns)
    direction = model.predict(row)[0]
    return "Up" if direction == 1 else "Down"

gr.Interface(
    fn=predict_stock_movement,
    inputs=[
        gr.Number(label="Open Price"),
        gr.Number(label="High Price"),
        gr.Number(label="Low Price"),
        gr.Number(label="Close Price"),
        gr.Number(label="Volume"),
        gr.Number(label="5D Volatility"),
        gr.Number(label="5D Momentum")
    ],
    outputs="text",
    title="Stock Movement Predictor"
).launch()
